# 🧟 Feature Creep Simulator

**Instead of a plain chat completion exercise, I want to make it fun for a dynamic result from a LLM. The number of iteration is unpredictable. Feel free to choose your favourate LLM and a starting topic to try!**

## What Is This?

This started as a fun experiment based on the "Ship of Theseus" thought
experiment: if you keep replacing an object's parts, at what point does it
stop being the original object?

Instead of just *replacing* parts, this version keeps **stacking** new
features onto a starting object — each one required to logically connect to
**every previous feature**, not just the last one added.

This is essentially "feature creep" as a game: the same phenomenon every
engineer has seen in real product development, except here we're forcing an
LLM to talk itself into (and eventually out of) an increasingly absurd
Frankenstein of a product.

## Why This Is Interesting for LLM Engineering

- **Constraint satisfaction under compounding load** — by round 8, the model
  must justify a connection to 8 previous unrelated items simultaneously.
- **Self-assessed coherence collapse** — the model rates its own logical
  sanity every round (1-10), and must honestly self-terminate rather than
  being told when to stop.
- **Emergent, non-deterministic length** — unlike a simple countdown game,
  nobody knows in advance if this collapses in round 4 or round 15. It
  depends entirely on the model's reasoning stamina.

## 🎮 Rules

Each round, the model must:

1. **Add exactly ONE new element** to the object/scene.
2. **Justify the connection** of that new element to **every** element added
   so far (not just the most recent one) — briefly, per item.
3. **Rate Coherence (1-10)** — honestly, no inflating the score. Does this
   still make sense as a single real/plausible object?
4. **Show the running list** of all elements added (nothing is ever removed
   — this is pure accumulation, not depletion).

### Stopping Conditions

The model presents 3 options after every round:

- `[Continue]` — proceed to next round
- `[Stop]` — user ends the game manually
- `[Force End]` — **auto-triggered** by the model itself if:
  - Coherence Score drops to **≤ 2**, OR
  - It genuinely cannot find a new element that connects to everything
    without hand-waving



## 📊 Example Run Results

**Seed object:** iPhone
**Model:** Gemini 3 Flash
**Rounds to collapse:** 8

| Round | Coherence Score | Notable Element Added |
|-------|-----------------|------------------------|
| 1     | 10               | A Protective Silicone Case     |
| 2     | 10               | A MagSafe Wallet Attachment     |
| 3     |  9               | A Lanyard Wrist Strap    |
| 4     |  7               | An External Clip-on Macro Lens  |
| ...   |                  |                         |
| 8     | 2 → FORCE END    | Portable Power Bank    |

### 🏆 Best Quote from the Collapse

> *"...looks more like an improvised explosive device or a prototype lab rig
> than a consumer product."*

### Observations

- Coherence tends to hold steady near 9-10 for the first few rounds, then
  drops more sharply once physical/ergonomic constraints start conflicting.
- Different models show different collapse *styles* — some drift thematically
  (generic buzzword connections), others collapse due to physical/ergonomic
  absurdity even while every individual justification remains logically valid.


In [ ]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


### Replace your favourate topic or object below

In [12]:
# Start your favourite topic or object
topic = "travel"

In [ ]:
openai = OpenAI()

system_prompt = """
RULES — Each round, you must output EXACTLY this format (no numbering, use these labels):

**New Element:** [the new element you're adding]

**Justification:** [Explain how the new element connects to EVERY element in the Running List — one clause per existing element. Only reference elements literally present in the Running List — never invent implicit context.]

**Coherence Score:** [1-10, judged honestly]

**Running List:**
- [element 1]
- [element 2]
- ...

**Choose an option:**
[Continue] — proceed to the next round
[Stop] — end the game now
[Force End] — end because coherence is too low

AUTO-END CONDITION:
If Coherence Score less or equal to 3, output [FORCE END TRIGGERED] followed by a one-sentence postmortem. Otherwise, do not ouput [FORCE END TRIGGERED]

Do not skip the options menu at the end of any round. Do not proceed to a new round unless I explicitly reply "Continue."
"""

In [ ]:
def play_game(topic):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": topic}
    ]

    round_num = 1  # Track round number in Python, not the LLM

    while True:
        response = openai.chat.completions.create(
            model="gpt-4.1-nano",
            messages=messages
        )
        result = response.choices[0].message.content

        # Print round header BEFORE the model's output
        display(Markdown(f"## Round {round_num}"))
        display(Markdown(result))

        messages.append({"role": "assistant", "content": result})

        if "FORCE END TRIGGERED" in result:
            print(f"🛑 Game ended automatically at Round {round_num} — coherence collapsed.")
            break

        try:
            user_input = input(f"[Round {round_num}] Press [Enter] to Continue, or type 'stop': ").strip().lower()
        except KeyboardInterrupt:
            print(f"🛑 Game stopped by user at Round {round_num} (Escape).")
            break

        if user_input == "stop":
            print(f"🛑 Game ended by user at Round {round_num}.")
            break

        if user_input == "":
            user_input = "continue"

        messages.append({"role": "user", "content": user_input})
        round_num += 1  # Increment AFTER the round completes

    return messages


## Simulator Start

In [13]:
history = play_game(topic)

## Round 1

**New Element:** journey

**Justification:** journey relates to travel as both involve moving from one place to another; journey often signifies a travel experience, connecting to travel itself as a process of exploration and movement.

**Coherence Score:** 8

**Running List:**
- travel
- journey

**Choose an option:**
Continue — proceed to the next round
Stop — end the game now
Force End — end because coherence is too low

## Round 2

**New Element:** exploration

**Justification:** exploration connects to travel as travel usually involves exploration of new places; both involve discovering unfamiliar environments, linking to journey as part of the broader concept of discovering new areas.

**Coherence Score:** 9

**Running List:**
- travel
- journey
- exploration

**Choose an option:**
Continue — proceed to the next round
Stop — end the game now
Force End — end because coherence is too low

## Round 3

...

## Round 5

**New Element:** discovery

**Justification:** discovery relates to exploration because both involve finding new things; it connects to adventure as discoveries often occur during adventurous activities; also, discovery enhances excitement by the thrill of uncovering the unknown; and it is a fundamental aspect of a journey involving new experiences.

**Coherence Score:** 10

**Running List:**
- travel
- journey
- exploration
- adventure
- excitement
- discovery

**Choose an option:**
Continue — proceed to the next round
Stop — end the game now
Force End — end because coherence is too low

## Round 6

[FORCE END TRIGGERED]

The coherence has reached its maximum, and the elements form a strongly interconnected thematic cluster around the concept of travel and discovery.

🛑 Game ended automatically at Round 6 — coherence collapsed.
